<h2>text processing — tokenization, stemming, lemmatization</h2>

<h3>building a regex word tokenizer</h3>

In [1]:
import re 

def tokenize(text): 
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+|[^\sA-Za-z0-9]", text)

In [3]:
tokenize("We are all stardust and stories.")

['We', 'are', 'all', 'stardust', 'and', 'stories', '.']

<h3>building a porter stemmer</h3>

In [7]:
def stem(word):
    if word.endswith("sses"):
        return word[:-2]
    if word.endswith("ies"):
        return word[:-2]
    if word.endswith("ss"):
        return word
    if word.endswith("s") and len(word) > 1:
        return word[:-1]
    return word

In [13]:
[stem(w) for w in ["caresses", "ponies", "caress", "cats", "bus"]]

['caress', 'poni', 'caress', 'cat', 'bu']

<h3>building a lookup-based lemmatizer</h3>

In [17]:
#lemmatization reduces a word to its dictionary form using grammar knowledge 
LEMMA_TABLE = {
    ("running", "VERB"): "run",
    ("ran", "VERB"): "run",
    ("runs", "VERB"): "run",
    ("better", "ADJ"): "good",
    ("best", "ADJ"): "good",
    ("cats", "NOUN"): "cat",
    ("cat", "NOUN"): "cat",
    ("were", "VERB"): "be",
    ("was", "VERB"): "be",
    ("is", "VERB"): "be",
}

def lemmatize(word, pos):
    key = (word.lower(), pos)
    if key in LEMMA_TABLE:
        return LEMMA_TABLE[key]
    if pos == "VERB" and word.endswith("ing"):
        return word[:-3]
    if pos == "NOUN" and word.endswith("s"):
        return word[:-1]
    return word.lower()

In [21]:
lemmatize("running", "VERB") 

'run'

In [23]:
lemmatize("cats", "NOUN")   

'cat'

In [27]:
lemmatize("better", "ADJ")  

'good'

In [29]:
lemmatize("watched", "VERB")

'watched'

In [31]:
def preprocess(text, pos_tagger=None):
    tokens = tokenize(text)
    stems = [stem(t.lower()) for t in tokens]
    tags = pos_tagger(tokens) if pos_tagger else [(t, "NOUN") for t in tokens]
    lemmas = [lemmatize(word, pos) for word, pos in tags]
    return {"tokens": tokens, "stems": stems, "lemmas": lemmas}

<h3>using NLTK</h3>

In [40]:
import nltk
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger_eng")

from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

text = "Occasionally, Fate pulls itself together again and Time is always waiting."
tokens = word_tokenize(text)
stems = [PorterStemmer().stem(t) for t in tokens]
lemmatizer = WordNetLemmatizer()
tagged = pos_tag(tokens)


def nltk_pos_to_wordnet(tag):
    if tag.startswith("V"):
        return "v"
    if tag.startswith("J"):
        return "a"
    if tag.startswith("R"):
        return "r"
    return "n"


lemmas = [lemmatizer.lemmatize(t, nltk_pos_to_wordnet(tag)) for t, tag in tagged]

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LALITHA\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\LALITHA\AppData\Roaming\nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\LALITHA\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


<h3>using spaCy</h3>

In [52]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp("Occasionally, Fate pulls itself together again and Time is always waiting.")

for token in doc:
    print(token.text, token.lemma_, token.pos_)

Occasionally occasionally ADV
, , PUNCT
Fate Fate PROPN
pulls pull VERB
itself itself PRON
together together ADV
again again ADV
and and CCONJ
Time Time PROPN
is be AUX
always always ADV
waiting wait VERB
. . PUNCT


<h2>text representation</h2>

<h3>building a bag of words & tf-idf</h3>

In [58]:
def build_vocab(docs):
    vocab = {}
    for doc in docs:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab 

def bag_of_words(docs, vocab):
    matrix = [[0] * len(vocab) for _ in docs]
    for i, doc in enumerate(docs):
        for token in doc:
            if token in vocab:
                matrix[i][vocab[token]] += 1
    return matrix

In [66]:
docs = [["dog", "hell", "oh", "other"], ["dog", "dog", "dear"]] 
vocab = build_vocab(docs) 

#[i][j] - how many times word j appears in doc i 
bag_of_words(docs, vocab)

[[1, 1, 1, 1, 0], [2, 0, 0, 0, 1]]

In [68]:
import math

#term frequency = word count/total no. of words 
def term_frequency(doc_bow, doc_length):
    return [c / doc_length if doc_length else 0 for c in doc_bow]

#document frequency = in how many docs this word appears 
def document_frequency(bow_matrix):
    df = [0] * len(bow_matrix[0])
    for row in bow_matrix:
        for j, count in enumerate(row):
            if count > 0:
                df[j] += 1
    return df

#inverse document frequency = log(total docs/docs which have the word) 
#common words - low importance, rare words - high importance 
def inverse_document_frequency(df, n_docs):
    return [math.log((n_docs + 1) / (d + 1)) + 1 for d in df] 

#freq in this doc & rare in other docs = imp word 
def tfidf(bow_matrix):
    n_docs = len(bow_matrix)
    df = document_frequency(bow_matrix)
    idf = inverse_document_frequency(df, n_docs)
    out = []
    for row in bow_matrix:
        length = sum(row)
        tf = term_frequency(row, length)
        out.append([tf_j * idf_j for tf_j, idf_j in zip(tf, idf)])
    return out

In [71]:
docs = [["the", "dog", "blue"],
        ["the", "eyes", "blue"],
        ["the", "dog", "horse"],] 
vocab = build_vocab(docs) 
bow = bag_of_words(docs, vocab) 
tfidf(bow)

[[0.3333333333333333, 0.42922735748392693, 0.42922735748392693, 0.0, 0.0],
 [0.3333333333333333, 0.0, 0.42922735748392693, 0.5643823935199818, 0.0],
 [0.3333333333333333, 0.42922735748392693, 0.0, 0.0, 0.5643823935199818]]

In [73]:
def l2_normalize(matrix):
    out = []
    for row in matrix:
        norm = math.sqrt(sum(x * x for x in row))
        out.append([x / norm if norm else 0 for x in row])
    return out

<h3>hybrid - TF-IDF weighted embeddings</h3>

In [79]:
def tfidf_weighted_embedding(doc, tfidf_scores, embedding_table, dim):
    vec = [0.0] * dim
    total_weight = 0.0
    for token in doc:
        if token not in embedding_table or token not in tfidf_scores:
            continue
        weight = tfidf_scores[token]
        emb = embedding_table[token]
        for i in range(dim):
            vec[i] += weight * emb[i]
        total_weight += weight
    if total_weight == 0:
        return vec
    return [v / total_weight for v in vec]

<h3>using scikit-learn's built-ins</h3>  

In [76]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

docs = ["the dog sang on the stage", "the horse sang on the stage", "the dog danced"]

bow_vectorizer = CountVectorizer()
bow = bow_vectorizer.fit_transform(docs)
print(bow_vectorizer.get_feature_names_out())
print(bow.toarray())

tfidf_vectorizer = TfidfVectorizer()
tfidf = tfidf_vectorizer.fit_transform(docs)
print(tfidf.toarray().round(3))

['danced' 'dog' 'horse' 'on' 'sang' 'stage' 'the']
[[0 1 0 1 1 1 2]
 [0 0 1 1 1 1 2]
 [1 1 0 0 0 0 1]]
[[0.    0.395 0.    0.395 0.395 0.395 0.613]
 [0.    0.    0.492 0.374 0.374 0.374 0.581]
 [0.72  0.548 0.    0.    0.    0.    0.425]]


<h2>word embeddings</h2>

<h3>word2vec from scratch</h3>